# ✈️ 수원 군공항 이전이 인근 부동산 가격에 미친 영향

> **이중차분(DID) + 합성통제(Synthetic Control) + Event Study**로 다각도 추정

**Research Question**: 수원 군공항 이전 결정(2025) 이후 처리지역(영통·팔달구) 아파트 가격은
통제지역(광명·안양) 대비 얼마나 변했는가?

**식별 전략**:
1. TWFE DID — 표준 추정
2. Event Study — 동태적 효과
3. Goodman-Bacon — 가중치 진단
4. Synthetic Control — 강건성 점검

## 1. 환경 설정

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from airport_di import DIDAnalysis, load_panel
from airport_di.data_loader import load_synthetic_panel, CASE_CONFIGS
from airport_di.visualization import (
    plot_outcome_timeseries, plot_event_study,
    plot_synthetic_control, plot_placebo_test, plot_dose_response
)

case = CASE_CONFIGS['suwon']
print(f'Case: {case.name}')
print(f'Treatment year: {case.treatment_year}')
print(f'Treatment regions: {case.treatment_regions}')
print(f'Control regions: {case.control_regions}')

## 2. 패널 데이터 로드 (실데이터 또는 합성)

In [ ]:
# 실데이터가 있으면
# panel = load_panel('suwon')

# 또는 합성 시뮬레이션
panel = load_synthetic_panel(
    n_treated=2,
    n_control=10,
    pre_periods=60,
    post_periods=24,
    treatment_effect=-0.05,  # 5% 가격 하락
)
panel.head()

## 3. 시계열 비교 — 평행 추세 시각 점검

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
plot_outcome_timeseries(panel, 'outcome', treatment_year=2025, ax=ax)
ax.set_title('처리집단 vs 통제집단 — 가격 추이')

## 4. TWFE DID 추정

In [ ]:
did = DIDAnalysis(
    panel,
    outcome='outcome',
    unit_col='region_code',
    time_col='period',
    treat_col='treatment',
    post_col='post',
)

twfe = did.run_twfe(cluster_se=True)
print(twfe.summary())

## 5. 평행 추세 검정

In [ ]:
pt = did.test_parallel_trends()
print(f"F-stat: {pt['F_stat']:.3f}, p={pt['p_value']:.4f}, pass={pt['pass']}")

## 6. Event Study — 동태적 효과

In [ ]:
es = did.run_event_study(leads=4, lags=8)
print(es.summary())

fig, ax = plt.subplots(figsize=(12, 6))
plot_event_study(es.raw_result.coef_df, ax=ax)

## 7. Goodman-Bacon 분해

In [ ]:
bacon = did.goodman_bacon_decomposition()
bacon

## 8. 합성통제 (강건성 점검)

In [ ]:
sc = did.run_synthetic_control(
    treated_unit='41110',  # 영통구
    donor_pool=case.control_regions,
)
print(sc.summary())

## 9. 종합 결론

**예상 시나리오**: 수원 군공항 이전 결정 발표 직후 처리지역 가격이 약 3~7% 상승.
이전 사례(대구) 분석에서는 **소음 노출 해소 + 재개발 기대감**이 주된 메커니즘.

**정책적 함의**:
- 군공항 이전은 인근 부동산에 양(+)의 외부효과 — 보상 기준 재산정 필요
- 신공항 입지 결정 시 음(-)의 효과 발생 → 사전 보상 체계 설계 필요